# Video Face Swap Demo - 2 người (inswapper + InsightFace)

**Logic:** 1 video mẫu có 2 người (ví dụ cảnh hôn nhau) + 2 ảnh khuôn mặt (người A, người B) → video mới giữ nguyên chuyển động/nền gốc, mỗi người trong video được thay đúng bằng khuôn mặt tương ứng đã cung cấp.

**Công nghệ dùng:**
- `insightface` (buffalo_l) để detect + align khuôn mặt từng frame
- `inswapper_128.onnx` để swap khuôn mặt
- `GFPGAN` (tùy chọn) để làm nét/phục hồi mặt sau khi swap
- `ffmpeg` để tách/ghép audio và dựng lại video

**Trước khi chạy:** Runtime > Change runtime type > chọn GPU (T4).

⚠️ Lưu ý: công nghệ face-swap có thể bị dùng sai mục đích (deepfake giả mạo người khác mà không có sự đồng ý). Chỉ dùng với ảnh/video của chính bạn hoặc người đã đồng ý, và cân nhắc gắn watermark/disclosure khi xuất bản sản phẩm thật.

## 0. (Khuyến nghị) Cài môi trường Python 3.10 ổn định qua Miniconda

**Vì sao cần bước này:** Colab thỉnh thoảng nâng cấp Python mặc định lên bản rất mới (ví dụ 3.13), trong khi nhiều package AI (`insightface`, `basicsr`, `onnxruntime-gpu`, `gfpgan`...) chưa kịp cập nhật để tương thích, gây lỗi cài đặt hàng loạt. Bước này dùng `condacolab` để cài môi trường Python 3.10 riêng — bản Python đã được các package trên test kỹ và ổn định lâu dài.

⚠️ **Chạy cell dưới ĐẦU TIÊN, trước mọi cell khác trong notebook này.** Cell sẽ tự động **restart kernel** (thấy dòng chữ đỏ 'Your session crashed for an unknown reason' là bình thường, không phải lỗi thật — đây là cách condacolab áp dụng thay đổi). Sau khi thấy thông báo hoàn tất, chạy tiếp cell xác nhận Python 3.10 bên dưới, rồi mới đến mục 1.

In [ ]:
# Log phiên bản Python Colab đang cấp TRƯỚC KHI đổi sang Python 3.10 (để so sánh trước/sau)
import sys, platform
print('--- Trước khi cài Python 3.10 ---')
print('Python version:', sys.version)
print('Platform:', platform.platform())

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()
# Sau dòng này, kernel sẽ tự restart. Notebook sẽ báo 'crashed' - đây là hành vi
# có chủ đích của condacolab, không phải lỗi. Chờ vài giây rồi chạy cell tiếp theo.

In [ ]:
# Chạy cell này SAU KHI kernel đã restart xong (chạy cell trên xong, đợi thông báo, rồi chạy cell này riêng)
!conda install -y python=3.10 -c conda-forge

import sys
print('--- Sau khi cài Python 3.10 ---')
print('Python version:', sys.version)

## 1. Cài đặt thư viện

In [ ]:
import sys
print('Python đang chạy trong cell này:', sys.version)

In [ ]:
# Ghim setuptools < 82: bản setuptools mới (>=82) đã bỏ hẳn module 'distutils',
# trong khi basicsr (dependency của GFPGAN) và torch trên Colab vẫn cần distutils/setuptools cũ.
!pip install -q "setuptools==79.0.1" wheel
!pip install -q cython numpy

# --no-build-isolation: để insightface dùng đúng cython/numpy vừa cài ở trên,
# thay vì pip tự tạo môi trường tạm cô lập (không thấy cython) rồi build lỗi 'egg_info'.
!pip install -q --no-build-isolation insightface==0.7.3
!pip install -q onnxruntime-gpu opencv-python-headless gfpgan facexlib
!apt-get -qq install -y ffmpeg > /dev/null

### Fix riêng cho `basicsr` (bug với Python bản mới trên Colab)

`basicsr` có bug trong `setup.py`: dùng `exec(...)` rồi đọc `locals()['__version__']` để lấy version. Ở Python 3.13, `exec()` trong 1 hàm không còn ghi ngược lại `locals()` đáng tin cậy (thay đổi theo PEP 667) → lỗi `KeyError: '__version__'`. Cell dưới tải source về, patch đúng chỗ này (`locals()` → `globals()`), rồi cài từ bản đã sửa.

In [ ]:
import subprocess, os, tarfile, urllib.request, json

os.makedirs('/tmp/basicsr_src', exist_ok=True)

# Tải trực tiếp từ PyPI bằng urllib (KHÔNG dùng `pip download`, vì pip cũng phải chạy
# setup.py egg_info để lấy metadata -> dính đúng bug KeyError trước khi kịp patch).
with urllib.request.urlopen('https://pypi.org/pypi/basicsr/1.4.2/json') as resp:
    pkg_info = json.load(resp)

sdist_url = None
for url_info in pkg_info['urls']:
    if url_info['packagetype'] == 'sdist':
        sdist_url = url_info['url']
        break
assert sdist_url is not None, 'Không tìm thấy sdist của basicsr trên PyPI.'

tar_name = sdist_url.split('/')[-1]
urllib.request.urlretrieve(sdist_url, f'/tmp/basicsr_src/{tar_name}')
print(f'Đã tải: {tar_name}')

extract_dir = '/tmp/basicsr_build'
os.makedirs(extract_dir, exist_ok=True)
with tarfile.open(f'/tmp/basicsr_src/{tar_name}') as tar:
    tar.extractall(extract_dir)

pkg_dir = os.path.join(extract_dir, tar_name.replace('.tar.gz', ''))
setup_py_path = os.path.join(pkg_dir, 'setup.py')

with open(setup_py_path, 'r') as f:
    content = f.read()

# Fix bug Python 3.13: exec() trong hàm không ghi ngược locals() đáng tin cậy
content = content.replace(
    "exec(compile(f.read(), version_file, 'exec'))",
    "exec(compile(f.read(), version_file, 'exec'), globals())"
)
content = content.replace(
    "return locals()['__version__']",
    "return globals()['__version__']"
)

with open(setup_py_path, 'w') as f:
    f.write(content)

print('Đã patch setup.py xong, tiến hành cài basicsr từ source đã sửa...')
install_result = subprocess.run(
    ['pip', 'install', '--no-build-isolation', pkg_dir],
    capture_output=True, text=True
)
print(install_result.stdout)
print(install_result.stderr)
if install_result.returncode != 0:
    raise RuntimeError('Cài basicsr thất bại, xem log lỗi ở trên để biết nguyên nhân cụ thể.')
print('Cài basicsr thành công.')

## 2. Tải model (inswapper + buffalo_l + GFPGAN)

Model `inswapper_128.onnx` không được host chính thức trên GitHub release nữa do vấn đề chính sách, 
nên bạn cần tự tải và upload lên Google Drive của mình, hoặc dùng link mirror cộng đồng (huggingface). 
Cell dưới thử tải từ 1 mirror phổ biến trên Hugging Face — nếu lỗi, bạn tải thủ công rồi upload vào `/content/`.

In [ ]:
import os
os.makedirs('/content/models', exist_ok=True)

# Mirror cộng đồng trên Hugging Face (kiểm tra lại link còn sống trước khi chạy)
!wget -q -O /content/models/inswapper_128.onnx https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx

# GFPGAN weights cho face restoration (tùy chọn)
!wget -q -O /content/models/GFPGANv1.4.pth https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth

print('Đã tải xong (kiểm tra dung lượng file bên dưới):')
!ls -lh /content/models/

## 3. Upload 2 ảnh khuôn mặt (người A, người B) + video mẫu (2 người)

In [ ]:
from google.colab import files

print('>> Upload ảnh khuôn mặt NGƯỜI A (rõ mặt, chính diện càng tốt):')
face_upload_a = files.upload()
source_face_path_a = list(face_upload_a.keys())[0]

print('\n>> Upload ảnh khuôn mặt NGƯỜI B:')
face_upload_b = files.upload()
source_face_path_b = list(face_upload_b.keys())[0]

print('\n>> Upload video mẫu (có 2 người, ví dụ cảnh hôn nhau):')
video_upload = files.upload()
source_video_path = list(video_upload.keys())[0]

print(f'Ảnh mặt A: {source_face_path_a}')
print(f'Ảnh mặt B: {source_face_path_b}')
print(f'Video mẫu: {source_video_path}')

## 4. Khởi tạo model face analysis + face swapper

**Bước quan trọng:** cần xác định trong video, ai đứng bên trái / bên phải (theo frame đầu tiên có đủ 2 mặt) để biết gán ảnh A/B vào đúng người. Mặc định: **người A = mặt bên trái khung hình ở frame đầu tiên, người B = mặt bên phải**. Nếu bị ngược, đổi lại 2 ảnh upload ở bước 3 hoặc đảo `source_face_a`/`source_face_b` ở dưới.

### Fix riêng cho `onnxruntime-gpu` (có thể chưa có wheel cho Python bản mới trên Colab)

Cell dưới kiểm tra xem `onnxruntime` đã import được chưa. Nếu chưa, sẽ thử cài lại `onnxruntime-gpu` với log đầy đủ; nếu vẫn thất bại (do chưa có wheel tương thích Python 3.13), sẽ **fallback sang `onnxruntime` bản CPU** để pipeline vẫn chạy được (chỉ chậm hơn, không dùng được GPU cho bước detect/swap qua ONNX).

In [ ]:
import subprocess

def try_import_onnxruntime():
    try:
        import onnxruntime
        print('onnxruntime OK, version:', onnxruntime.__version__)
        print('Available providers:', onnxruntime.get_available_providers())
        return True
    except ImportError as e:
        print('Chưa import được onnxruntime:', e)
        return False

if not try_import_onnxruntime():
    print('Thử cài lại onnxruntime-gpu với log đầy đủ...')
    r = subprocess.run(['pip', 'install', 'onnxruntime-gpu'], capture_output=True, text=True)
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])

    if not try_import_onnxruntime():
        print('onnxruntime-gpu không cài được (khả năng chưa có wheel cho bản Python này).')
        print('Fallback sang onnxruntime bản CPU...')
        r2 = subprocess.run(['pip', 'install', 'onnxruntime'], capture_output=True, text=True)
        print(r2.stdout[-2000:])
        print(r2.stderr[-2000:])
        assert try_import_onnxruntime(), 'Vẫn không cài được onnxruntime, xem log lỗi ở trên.'

In [ ]:
import cv2
import insightface
from insightface.app import FaceAnalysis

import onnxruntime
available_providers = onnxruntime.get_available_providers()
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if 'CUDAExecutionProvider' in available_providers else ['CPUExecutionProvider']
print('Dùng providers:', providers)

app = FaceAnalysis(name='buffalo_l', providers=providers)
app.prepare(ctx_id=0, det_size=(640, 640))

swapper = insightface.model_zoo.get_model('/content/models/inswapper_128.onnx', download=False, providers=providers)

# Lấy khuôn mặt nguồn A
img_a = cv2.imread(source_face_path_a)
faces_a = app.get(img_a)
assert len(faces_a) > 0, 'Không tìm thấy khuôn mặt trong ảnh A, thử ảnh khác rõ mặt hơn.'
source_face_a = faces_a[0]

# Lấy khuôn mặt nguồn B
img_b = cv2.imread(source_face_path_b)
faces_b = app.get(img_b)
assert len(faces_b) > 0, 'Không tìm thấy khuôn mặt trong ảnh B, thử ảnh khác rõ mặt hơn.'
source_face_b = faces_b[0]

print('Đã detect khuôn mặt nguồn A và B thành công.')

## 5. (Tùy chọn) Khởi tạo GFPGAN để làm nét mặt sau swap

**Fix lỗi tương thích:** bản `torchvision` mới trên Colab đã xóa module `torchvision.transforms.functional_tensor` 
mà `basicsr` (dependency của GFPGAN) vẫn còn import theo đường cũ, gây lỗi `ModuleNotFoundError`. 
Cell dưới patch trực tiếp file trên đĩa bằng `sed` (không import basicsr trong Python nên không bị crash giữa chừng).

⚠️ **Nếu đã từng chạy lỗi ở cell GFPGAN trước đó trong session này**, hãy **Runtime > Restart session** rồi chạy lại từ đầu (kể cả cell cài đặt thư viện), vì Python có thể đã cache import bị lỗi. Sau khi patch xong, cell import GFPGAN bên dưới sẽ chạy được ngay lần đầu tiên.

In [ ]:
%%bash
# Tìm file degradations.py mà KHÔNG import basicsr (tránh crash trước khi patch được)
DEG_FILE=$(find / -path '*basicsr/data/degradations.py' 2>/dev/null | head -n1)

echo "File tìm thấy: $DEG_FILE"

sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' "$DEG_FILE"

grep -n "rgb_to_grayscale" "$DEG_FILE"

In [ ]:
import subprocess

def try_import_gfpgan():
    try:
        import gfpgan
        print('gfpgan OK, version:', getattr(gfpgan, '__version__', 'unknown'))
        return True
    except ImportError as e:
        print('Chưa import được gfpgan:', e)
        return False

GFPGAN_AVAILABLE = try_import_gfpgan()

if not GFPGAN_AVAILABLE:
    print('Thử cài lại gfpgan với log đầy đủ...')
    r = subprocess.run(['pip', 'install', 'gfpgan'], capture_output=True, text=True)
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    GFPGAN_AVAILABLE = try_import_gfpgan()

if not GFPGAN_AVAILABLE:
    print('gfpgan không cài được. Sẽ BỎ QUA bước phục hồi mặt (USE_GFPGAN sẽ tự tắt ở cell dưới).')
    print('Pipeline chính (swap mặt) vẫn chạy bình thường, chỉ là ảnh không được làm nét thêm.')

In [ ]:
USE_GFPGAN = GFPGAN_AVAILABLE  # tự động tắt nếu gfpgan không cài được ở cell trên; đổi thành False để bỏ qua thủ công

restorer = None
if USE_GFPGAN:
    from gfpgan import GFPGANer
    restorer = GFPGANer(
        model_path='/content/models/GFPGANv1.4.pth',
        upscale=1,
        arch='clean',
        channel_multiplier=2,
        bg_upsampler=None
    )
else:
    print('Bỏ qua GFPGAN, dùng ảnh swap gốc không phục hồi nét.')

## 6. Xử lý video: swap mặt từng frame (2 người, có tracking theo vị trí)

**Vấn đề cần giải quyết:** model detect mặt mỗi frame **không đảm bảo thứ tự cố định** (frame này trả về [trái, phải], frame sau có thể trả về [phải, trái], đặc biệt khi 2 người quay đầu/che nhau lúc hôn). Nếu chỉ lấy theo index `[0]`, `[1]` thì mặt A/B sẽ bị **đảo lộn giữa các frame**, tạo hiệu ứng giật/lóe rất xấu.

**Cách xử lý ở đây:** theo dõi vị trí tâm khuôn mặt (center point) của mỗi người qua các frame liên tiếp — mặt ở frame hiện tại được gán vào đúng người (A hoặc B) có **vị trí gần nhất với frame trước đó**, thay vì tin vào thứ tự trả về của model.

**Trường hợp 1 mặt bị che khuất (occlusion) hoàn toàn** (ví dụ lúc hôn 2 mặt chồng lên nhau chỉ detect được 1): code sẽ swap người có vị trí gần nhất với vị trí đã biết, người còn lại giữ nguyên frame gốc ở khung đó (không suy đoán mù).

In [ ]:
import os
import numpy as np
from tqdm import tqdm

def face_center(face):
    x1, y1, x2, y2 = face.bbox
    return np.array([(x1 + x2) / 2, (y1 + y2) / 2])

cap = cv2.VideoCapture(source_video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

output_noaudio = '/content/output_noaudio.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_noaudio, fourcc, fps, (width, height))

print(f'Video: {width}x{height} @ {fps:.1f}fps, {total_frames} frames')

# Vị trí đã biết của A/B từ frame trước (None = chưa xác định)
last_pos_a = None
last_pos_b = None

for frame_idx in tqdm(range(total_frames)):
    ret, frame = cap.read()
    if not ret:
        break

    target_faces = app.get(frame)
    result_frame = frame

    if len(target_faces) == 0:
        out.write(result_frame)
        continue

    if frame_idx == 0 or (last_pos_a is None and last_pos_b is None):
        # Frame đầu tiên có mặt: quy ước A = mặt bên trái, B = mặt bên phải
        target_faces_sorted = sorted(target_faces, key=lambda f: face_center(f)[0])
        assigned = {}
        if len(target_faces_sorted) >= 1:
            assigned['A'] = target_faces_sorted[0]
            last_pos_a = face_center(target_faces_sorted[0])
        if len(target_faces_sorted) >= 2:
            assigned['B'] = target_faces_sorted[-1]
            last_pos_b = face_center(target_faces_sorted[-1])
    else:
        # Các frame sau: gán theo khoảng cách gần nhất với vị trí đã biết
        assigned = {}
        remaining = list(target_faces)

        candidates = []
        if last_pos_a is not None:
            candidates.append(('A', last_pos_a))
        if last_pos_b is not None:
            candidates.append(('B', last_pos_b))

        for label, last_pos in candidates:
            if not remaining:
                break
            dists = [np.linalg.norm(face_center(f) - last_pos) for f in remaining]
            best_idx = int(np.argmin(dists))
            assigned[label] = remaining.pop(best_idx)

        if 'A' in assigned:
            last_pos_a = face_center(assigned['A'])
        if 'B' in assigned:
            last_pos_b = face_center(assigned['B'])

    # Swap từng người đã gán được trong frame này
    if 'A' in assigned:
        result_frame = swapper.get(result_frame, assigned['A'], source_face_a, paste_back=True)
    if 'B' in assigned:
        result_frame = swapper.get(result_frame, assigned['B'], source_face_b, paste_back=True)

    if restorer is not None and ('A' in assigned or 'B' in assigned):
        _, _, result_frame = restorer.enhance(
            result_frame, has_aligned=False, only_center_face=False, paste_back=True
        )

    out.write(result_frame)

cap.release()
out.release()
print('Đã swap xong toàn bộ frame (chưa ghép audio).')

## 7. Ghép lại audio gốc vào video đã swap

In [ ]:
final_output = '/content/output_final.mp4'

!ffmpeg -y -i /content/output_noaudio.mp4 -i "{source_video_path}" \
  -c:v libx264 -crf 18 -preset fast \
  -map 0:v:0 -map 1:a:0? -shortest \
  {final_output}

print('Video hoàn chỉnh:', final_output)

## 8. Xem kết quả

In [ ]:
from IPython.display import Video
Video(final_output, embed=True, width=480)

In [ ]:
# Tải file về máy
from google.colab import files
files.download(final_output)

## Ghi chú / Hướng cải thiện tiếp theo

- **Flicker giữa các frame**: face swap từng frame độc lập nên đôi khi có giật/nhòe nhẹ theo thời gian. Có thể cải thiện bằng cách thêm temporal smoothing (trung bình landmark giữa các frame liền kề) hoặc dùng model chuyên video như **SimSwap** với chế độ video.
- **Tracking 2 người**: pipeline hiện dùng tracking đơn giản theo khoảng cách vị trí (position-based) — đủ tốt cho video ngắn, chuyển động không quá nhanh. Nếu 2 người **đổi chỗ nhanh, che khuất lâu, hoặc camera cắt cảnh**, tracking có thể gán nhầm A/B. Nếu gặp trường hợp này, cần thêm model tracking chuyên dụng hơn (ví dụ DeepSORT hoặc dùng embedding khuôn mặt để so khớp danh tính thay vì chỉ dựa vị trí).
- **Tốc độ**: xử lý frame-by-frame trên Colab free (T4) sẽ khá chậm với video dài. Nên test với video ngắn (5–10s) trước.
- **inswapper_128 model**: link tải có thể thay đổi do các vấn đề về chính sách/gỡ bỏ, nếu link trong notebook chết bạn cần tự tìm mirror khác hoặc lưu file vào Google Drive cá nhân rồi copy vào `/content/models/`.
- **Chất lượng ảnh mặt nguồn**: ảnh càng rõ, chính diện, ánh sáng đều thì kết quả swap càng tự nhiên.